# Part A — Introduction & Conceptual Framework  
## Descriptive analysis of the EU fossil-fuel power plant sample

This notebook is designed for **Section a** of the report: **Introduction & Conceptual Framework**.

### Objectives
This notebook helps you produce:
- the **descriptive statistics** needed to present the sample,
- the **core contextual facts** about the EU fossil-fuel power sector,
- a small set of **clean appendix-ready figures**,
- the numerical elements needed to write the introductory paragraphs.

### Main dataset used
According to the workflow and the database overview, **Part a** relies primarily on:

- `data/processed/gppd_eu_metrics.csv`

This file contains the plant-level information needed to describe:
- the number of plants,
- the installed capacity,
- the fuel mix,
- the plant age structure,
- the carbon intensity profile,
- the remaining lifetime structure,
- the geographic distribution of the sample.

### Suggested use in the report
For Section **a**, you will mostly use this notebook to extract:
1. a short paragraph describing the sample,
2. 2–4 summary statistics,
3. possibly 1 very small descriptive figure in the appendix,
4. appendix figures and tables if needed.

> Note: this notebook is intentionally limited to **descriptive and contextual analysis**.  
> The scenario-based analysis belongs to later sections (`c`, `e`, etc.).


## 1. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Optional display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Paths
PROJECT_ROOT = Path.cwd()

# Adjust if needed:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "gppd_eu_metrics.csv"
FIG_DIR = PROJECT_ROOT / "figures" / "part_a"
TAB_DIR = PROJECT_ROOT / "results" / "part_a"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print("Expected data path:", DATA_PATH)
print("Figures will be saved in:", FIG_DIR)
print("Tables will be saved in:", TAB_DIR)


## 2. Load the plant-level dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()


### Expected key columns
The notebook assumes the following columns are available:
- `gppd_idnr`
- `name`
- `country`
- `country_long`
- `fuel`
- `capacity_mw`
- `commissioning_year`
- `age_2020_years`
- `lifetime`
- `remaining_life_2020_years`
- `carbon_intensity_tCO2_per_MWh_elec`
- `om_variable_eur2024_per_MWh`
- `capacity_factor_2020`
- `latitude`
- `longitude`


In [ ]:
expected_cols = [
    "gppd_idnr",
    "name",
    "country",
    "country_long",
    "fuel",
    "capacity_mw",
    "commissioning_year",
    "age_2020_years",
    "lifetime",
    "remaining_life_2020_years",
    "carbon_intensity_tCO2_per_MWh_elec",
    "om_variable_eur2024_per_MWh",
    "capacity_factor_2020",
    "latitude",
    "longitude",
]

missing_cols = [c for c in expected_cols if c not in df.columns]
print("Missing columns:", missing_cols if missing_cols else "None")


## 3. Quick data quality checks
These checks are useful for the appendix or for a short methodological note in the report.


In [ ]:
quality_summary = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_values": [df[c].isna().sum() for c in df.columns],
    "missing_share_pct": [100 * df[c].isna().mean() for c in df.columns],
})

quality_summary.sort_values("missing_share_pct", ascending=False).reset_index(drop=True)


In [ ]:
quality_summary.to_csv(TAB_DIR / "quality_summary.csv", index=False)
print("Saved quality summary.")


## 4. Core sample description for the introduction
This section produces the main numbers that can be directly reused in the text of Section a.


In [ ]:
n_plants = len(df)
n_countries = df["country"].nunique()
total_capacity_gw = df["capacity_mw"].sum() / 1000

sample_summary = {
    "Number of plants": n_plants,
    "Number of countries": n_countries,
    "Total installed capacity (GW)": round(total_capacity_gw, 2),
    "Average plant capacity (MW)": round(df["capacity_mw"].mean(), 2),
    "Median plant capacity (MW)": round(df["capacity_mw"].median(), 2),
    "Average age in 2020 (years)": round(df["age_2020_years"].mean(), 2),
    "Median age in 2020 (years)": round(df["age_2020_years"].median(), 2),
}

sample_summary


In [ ]:
sample_summary_df = pd.DataFrame(
    {"Metric": list(sample_summary.keys()), "Value": list(sample_summary.values())}
)
sample_summary_df.to_csv(TAB_DIR / "sample_summary.csv", index=False)
sample_summary_df


## 5. Fuel mix description
This is one of the most useful descriptive blocks for Section a.


In [ ]:
fuel_summary = (
    df.groupby("fuel", as_index=False)
      .agg(
          plants=("gppd_idnr", "count"),
          capacity_mw=("capacity_mw", "sum"),
          avg_capacity_mw=("capacity_mw", "mean"),
          avg_age_2020_years=("age_2020_years", "mean"),
          avg_carbon_intensity=("carbon_intensity_tCO2_per_MWh_elec", "mean"),
          avg_capacity_factor_2020=("capacity_factor_2020", "mean"),
      )
)

fuel_summary["capacity_gw"] = fuel_summary["capacity_mw"] / 1000
fuel_summary["plant_share_pct"] = 100 * fuel_summary["plants"] / fuel_summary["plants"].sum()
fuel_summary["capacity_share_pct"] = 100 * fuel_summary["capacity_mw"] / fuel_summary["capacity_mw"].sum()

fuel_summary = fuel_summary[
    [
        "fuel",
        "plants",
        "plant_share_pct",
        "capacity_gw",
        "capacity_share_pct",
        "avg_capacity_mw",
        "avg_age_2020_years",
        "avg_carbon_intensity",
        "avg_capacity_factor_2020",
    ]
].sort_values("capacity_gw", ascending=False)

fuel_summary


In [ ]:
fuel_summary.to_csv(TAB_DIR / "fuel_summary.csv", index=False)
print("Saved fuel summary.")


### Optional text-ready sentence generator
This cell creates a short paragraph draft that you can adapt in the report.


In [ ]:
fuel_cap = fuel_summary.set_index("fuel")["capacity_share_pct"].to_dict()
fuel_plants = fuel_summary.set_index("fuel")["plants"].to_dict()

draft_text = (
    f"The sample includes {n_plants} fossil-fuel power plants across {n_countries} countries, "
    f"representing {total_capacity_gw:.1f} GW of installed capacity. "
    f"Gas accounts for {fuel_cap.get('gas', 0):.1f}% of capacity, "
    f"coal for {fuel_cap.get('coal', 0):.1f}%, and oil for {fuel_cap.get('oil', 0):.1f}%. "
    f"In terms of plant count, the sample includes {fuel_plants.get('gas', 0)} gas plants, "
    f"{fuel_plants.get('coal', 0)} coal plants, and {fuel_plants.get('oil', 0)} oil plants."
)

print(draft_text)


## 6. Country-level descriptive statistics
This section is useful to identify the main countries represented in the sample.


In [ ]:
country_summary = (
    df.groupby(["country", "country_long"], as_index=False)
      .agg(
          plants=("gppd_idnr", "count"),
          capacity_mw=("capacity_mw", "sum")
      )
)

country_summary["capacity_gw"] = country_summary["capacity_mw"] / 1000
country_summary["capacity_share_pct"] = 100 * country_summary["capacity_mw"] / country_summary["capacity_mw"].sum()

country_summary = country_summary.sort_values("capacity_gw", ascending=False).reset_index(drop=True)
country_summary.head(15)


In [ ]:
country_summary.to_csv(TAB_DIR / "country_summary.csv", index=False)
print("Saved country summary.")


## 7. Vintage, age and remaining lifetime structure
This section helps justify why the power sector is particularly exposed to stranding risk.


In [ ]:
vintage_summary = pd.DataFrame({
    "avg_age_2020_years": [df["age_2020_years"].mean()],
    "median_age_2020_years": [df["age_2020_years"].median()],
    "avg_remaining_life_2020_years": [df["remaining_life_2020_years"].mean()],
    "median_remaining_life_2020_years": [df["remaining_life_2020_years"].median()],
    "share_past_end_of_life_pct": [100 * (df["remaining_life_2020_years"] < 0).mean()],
    "share_pre_2000_pct": [100 * (df["commissioning_year"] < 2000).mean()],
})
vintage_summary


In [ ]:
vintage_summary.to_csv(TAB_DIR / "vintage_summary.csv", index=False)
print("Saved vintage summary.")


## 8. Carbon intensity and baseline utilization
These indicators are directly relevant for the conceptual framework:
- carbon intensity matters because carbon pricing will affect variable costs,
- baseline capacity factor helps characterize plant utilization.


In [ ]:
carbon_util_summary = (
    df.groupby("fuel", as_index=False)
      .agg(
          mean_carbon_intensity=("carbon_intensity_tCO2_per_MWh_elec", "mean"),
          median_carbon_intensity=("carbon_intensity_tCO2_per_MWh_elec", "median"),
          mean_capacity_factor=("capacity_factor_2020", "mean"),
          median_capacity_factor=("capacity_factor_2020", "median"),
          mean_om_cost=("om_variable_eur2024_per_MWh", "mean")
      )
)
carbon_util_summary


In [ ]:
carbon_util_summary.to_csv(TAB_DIR / "carbon_util_summary.csv", index=False)
print("Saved carbon/intensity summary.")


## 9. Figures for Section a / appendix
For a short report, I recommend putting **most of these figures in the appendix**, not in the main text.


### Figure A1 — Installed capacity by fuel
This is one of the most useful descriptive figures for the appendix.


In [ ]:
fuel_plot = fuel_summary.sort_values("capacity_gw", ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(fuel_plot["fuel"], fuel_plot["capacity_gw"])
ax.set_title("Installed fossil-fuel capacity by fuel")
ax.set_xlabel("Fuel")
ax.set_ylabel("Capacity (GW)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig.savefig(FIG_DIR / "A1_capacity_by_fuel.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure A2 — Top countries by installed capacity
Useful if you want to briefly mention where the sample is concentrated.


In [ ]:
top_countries = country_summary.head(10)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(top_countries["country"], top_countries["capacity_gw"])
ax.set_title("Top 10 countries by installed fossil-fuel capacity")
ax.set_xlabel("Country")
ax.set_ylabel("Capacity (GW)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig.savefig(FIG_DIR / "A2_top10_capacity_by_country.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure A3 — Distribution of plant age
Useful to document the vintage structure of the sample.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["age_2020_years"].dropna(), bins=25)
ax.set_title("Distribution of plant age in 2020")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of plants")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig.savefig(FIG_DIR / "A3_age_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure A4 — Carbon intensity by fuel
A boxplot is useful to visualize the structural difference between coal, gas and oil.


In [ ]:
fuels = sorted(df["fuel"].dropna().unique())
data = [df.loc[df["fuel"] == f, "carbon_intensity_tCO2_per_MWh_elec"].dropna() for f in fuels]

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(data, tick_labels=fuels)
ax.set_title("Carbon intensity by fuel")
ax.set_xlabel("Fuel")
ax.set_ylabel("tCO$_2$/MWh$_{elec}$")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig.savefig(FIG_DIR / "A4_carbon_intensity_by_fuel.png", dpi=300, bbox_inches="tight")
plt.show()


### Figure A5 — Remaining lifetime distribution
This figure is relevant because stranding risk depends partly on the remaining life of the assets.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["remaining_life_2020_years"].dropna(), bins=25)
ax.set_title("Distribution of remaining technical lifetime in 2020")
ax.set_xlabel("Remaining lifetime (years)")
ax.set_ylabel("Number of plants")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()

fig.savefig(FIG_DIR / "A5_remaining_lifetime_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


### Optional Figure A6 — Geographic distribution of plants
This is useful for the appendix only. It is optional.


In [ ]:
if {"longitude", "latitude"}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(
        df["longitude"], 
        df["latitude"], 
        s=np.clip(df["capacity_mw"] / 50, 10, 200),
        alpha=0.6
    )
    ax.set_title("Geographic distribution of EU fossil-fuel plants")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.3)
    plt.tight_layout()

    fig.savefig(FIG_DIR / "A6_geographic_distribution.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Latitude/longitude not available.")


## 10. Export a compact text-ready table for the report
This table is useful if you want one short descriptive table in the appendix.


In [ ]:
appendix_table = fuel_summary.copy()
appendix_table.columns = [
    "Fuel",
    "Plants",
    "Plant share (%)",
    "Capacity (GW)",
    "Capacity share (%)",
    "Avg capacity (MW)",
    "Avg age in 2020 (years)",
    "Avg carbon intensity",
    "Avg capacity factor 2020",
]
appendix_table


In [ ]:
appendix_table.to_csv(TAB_DIR / "appendix_table_fuel_summary.csv", index=False)
print("Saved appendix-ready table.")


## 11. Suggested writing blocks for Section a
You can reuse the numbers generated above to write:

### (i) Sample paragraph
- number of plants,
- number of countries,
- total installed capacity,
- fuel composition.

### (ii) Why the sector is exposed to transition risk
- fossil-fuel plants are carbon-intensive,
- they have long technical lifetimes,
- the fleet includes older assets,
- carbon pricing directly affects variable costs.

### (iii) Why these plant characteristics matter for the model
- `carbon_intensity_tCO2_per_MWh_elec` links plants to carbon costs,
- `capacity_mw` is the key aggregation weight,
- `commissioning_year`, `lifetime`, and `remaining_life_2020_years` are relevant for stranding,
- `capacity_factor_2020` is useful for utilization-based robustness analysis.


## 12. Optional: automatic markdown summary for your report draft
This cell creates a compact descriptive block you can paste into your draft and then edit.


In [ ]:
avg_age = df["age_2020_years"].mean()
avg_remaining = df["remaining_life_2020_years"].mean()

summary_text = f'''
The plant-level sample contains {n_plants} fossil-fuel power plants located in {n_countries} European countries, for a total installed capacity of {total_capacity_gw:.1f} GW.
The sample is dominated by gas and coal plants in both plant count and installed capacity, while oil plays a smaller residual role.
The average plant age in 2020 is {avg_age:.1f} years, which highlights the importance of plant vintage in the analysis of stranding risk.
The dataset also provides plant-level carbon intensity, variable O\&M costs, and baseline capacity factors, which are key inputs for the downstream profitability and stranding analysis.
'''
print(summary_text)


## 13. Final note
For **Section a**, this notebook is sufficient to:
- characterize the sample,
- motivate the focus on fossil-fuel plants,
- explain why plant-level characteristics matter for transition risk,
- generate appendix-ready figures and tables.

The **scenario-based** and **profitability-based** analyses should be kept for later sections.
